# Stanford RNA 3D Folding Part 2 — Hybrid Biopython + Routed Protenix

This notebook is a cleaned, higher-upside hybrid built from your stronger **Biopython/Protenix** notebook and the better ideas from your **handup** notebook.

What changed:
- keep the **Biopython template alignment** and the **batched Protenix** inference path from the stronger notebook
- add **quality-based template routing** instead of blindly taking the top 5 templates
- use **better template coordinate adaptation** for insertions/deletions
- keep **chunked long-sequence inference** and make the chunk stitching safer
- add a hard **sanitization + clipping** pass so bad coordinates never leak into `submission.csv`

Expected use:
1. Attach the same Protenix dataset(s) you used for the Biopython notebook.
2. Run all cells.
3. Submit the generated `submission.csv`.

In [ ]:

import sys
import subprocess
from pathlib import Path

def _wheel_compatible(path: Path) -> bool:
    name = path.name.lower()
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    if name.endswith("none-any.whl"):
        return True
    if py_tag in name:
        return True
    if "abi3" in name and "linux" in name:
        return True
    if "py3-none" in name or "py2.py3-none" in name:
        return True
    return False

def _find_best_wheel(keywords):
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    wheels = []
    for p in root.rglob("*.whl"):
        name = p.name.lower()
        if all(k in name for k in keywords):
            wheels.append(p)
    if not wheels:
        return None

    def score(p: Path):
        name = p.name.lower()
        py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
        return (
            py_tag in name,
            "abi3" in name,
            "manylinux" in name,
            -len(str(p)),
            name
        )
    wheels.sort(key=score, reverse=True)
    return wheels[0]

def _maybe_install(label, import_name, keywords):
    try:
        __import__(import_name)
        print(f"{label}: already available")
        return
    except Exception:
        pass

    wheel = _find_best_wheel(keywords)
    if wheel is None:
        print(f"{label}: wheel not found under /kaggle/input (skipping)")
        return
    if not _wheel_compatible(wheel):
        print(f"{label}: found wheel but it looks incompatible with Python {sys.version_info.major}.{sys.version_info.minor}: {wheel.name}")
        return

    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(wheel)]
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=False)

_maybe_install("Biopython", "Bio", ["biopython"])
_maybe_install("biotite", "biotite", ["biotite"])
_maybe_install("RDKit", "rdkit", ["rdkit"])
_maybe_install("ml_collections", "ml_collections", ["ml", "collections"])


### Notes

- This notebook is designed for **Kaggle submission runtime**.
- It auto-searches for wheel files under `/kaggle/input` and installs them when available.
- It also tries to auto-find the Protenix root if the default path is missing.
- The last cell runs `main()` and writes `submission.csv`.

In [ ]:

import gc
import json
import math
import os
import random
import sys
import time
import traceback
import warnings

from pathlib import Path

os.environ["PROTENIX_CODE_DIR"] = "/kaggle/input/protenix-rmsa-repo/protenix_kaggle"
os.environ["PROTENIX_ROOT_DIR"] = "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"
sys.path.insert(0, "/kaggle/input/protenix-rmsa-repo/protenix_kaggle")
os.environ["LAYERNORM_TYPE"] = "torch" 

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm


warnings.filterwarnings("ignore")

# ───────────────────────── Paths & Constants ─────────────────────────────
COMP_DATA_DIR = "/kaggle/input/competitions/stanford-rna-3d-folding-2"
DEFAULT_TEST_CSV = f"{COMP_DATA_DIR}/test_sequences.csv"
DEFAULT_TRAIN_CSV = f"{COMP_DATA_DIR}/train_sequences.csv"
DEFAULT_TRAIN_LBLS = f"{COMP_DATA_DIR}/train_labels.csv"
DEFAULT_VAL_CSV = f"{COMP_DATA_DIR}/validation_sequences.csv"
DEFAULT_VAL_LBLS = f"{COMP_DATA_DIR}/validation_labels.csv"
DEFAULT_OUTPUT = "/kaggle/working/submission.csv"

MODEL_NAME = os.environ.get("MODEL_NAME", "protenix_base_20250630_v1.0.0")
DEFAULT_CODE_DIR = os.environ.get(
    "PROTENIX_CODE_DIR_DEFAULT",
    "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1",
)
DEFAULT_ROOT_DIR = os.environ.get("PROTENIX_ROOT_DIR_DEFAULT", DEFAULT_CODE_DIR)

N_SAMPLE = int(os.environ.get("N_SAMPLE", "5"))
SEED = int(os.environ.get("SEED", "42"))
MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", "512"))
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP", "128"))
MAX_LONG_SAMPLES = int(os.environ.get("MAX_LONG_SAMPLES", "3"))

# Template routing / ranking
MAX_TEMPLATE_SEARCH = int(os.environ.get("MAX_TEMPLATE_SEARCH", "12"))
MAX_TEMPLATE_CANDIDATES = int(os.environ.get("MAX_TEMPLATE_CANDIDATES", "12"))
TEMPLATE_ONLY_CONF = float(os.environ.get("TEMPLATE_ONLY_CONF", "0.78"))
TEMPLATE_STRONG_CONF = float(os.environ.get("TEMPLATE_STRONG_CONF", "0.65"))
TEMPLATE_MEDIUM_CONF = float(os.environ.get("TEMPLATE_MEDIUM_CONF", "0.58"))
TEMPLATE_WEAK_CONF = float(os.environ.get("TEMPLATE_WEAK_CONF", "0.50"))
MIN_TEMPLATE_QUALITY_KEEP = float(os.environ.get("MIN_TEMPLATE_QUALITY_KEEP", "0.32"))

QUALITY_DELTA = float(os.environ.get("QUALITY_DELTA", "0.10"))
DIVERSITY_LAMBDA = float(os.environ.get("DIVERSITY_LAMBDA", "0.030"))
DIVERSITY_CAP = float(os.environ.get("DIVERSITY_CAP", "3.50"))
RMSD_SUBSAMPLE = int(os.environ.get("RMSD_SUBSAMPLE", "500"))
AUGMENT_TEMPLATE_VARIANTS = int(os.environ.get("AUGMENT_TEMPLATE_VARIANTS", "3"))

SHORT_PTX_BASE_QUALITY = float(os.environ.get("SHORT_PTX_BASE_QUALITY", "0.67"))
LONG_PTX_BASE_QUALITY = float(os.environ.get("LONG_PTX_BASE_QUALITY", "0.61"))

COORD_CLIP_MIN = float(os.environ.get("COORD_CLIP_MIN", "-999.999"))
COORD_CLIP_MAX = float(os.environ.get("COORD_CLIP_MAX", "9999.999"))
ABSURD_COORD_THRESHOLD = float(os.environ.get("ABSURD_COORD_THRESHOLD", "100000.0"))

USE_PROTENIX = str(os.environ.get("USE_PROTENIX", "true")).strip().lower() not in {"0", "false", "no", "off"}

IS_KAGGLE = os.path.exists("/kaggle/input")
LOCAL_N_SAMPLES = None if IS_KAGGLE else 2

def parse_bool(value: str, default: bool = False) -> str:
    v = str(value).strip().lower()
    if v in {"1", "true", "t", "yes", "y", "on"}:
        return "true"
    if v in {"0", "false", "f", "no", "n", "off"}:
        return "false"
    return "true" if default else "false"

USE_MSA = parse_bool(os.environ.get("USE_MSA", "false"))
USE_TEMPLATE = parse_bool(os.environ.get("USE_TEMPLATE", "false"))
USE_RNA_MSA = parse_bool(os.environ.get("USE_RNA_MSA", "true"))

# ───────────────────────── General Utilities ─────────────────────────────
def seed_everything(seed: int) -> None:
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = True
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

def _auto_find_competition_data() -> str | None:
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    required = {
        "train_sequences.csv",
        "validation_sequences.csv",
        "test_sequences.csv",
        "train_labels.csv",
        "validation_labels.csv",
    }
    candidates = []
    for p in root.rglob("*"):
        if p.is_dir() and required.issubset({x.name for x in p.iterdir() if x.is_file()}):
            candidates.append(p)
    if not candidates:
        return None
    candidates.sort(key=lambda p: ("stanford-rna-3d-folding-2" not in str(p).lower(), len(str(p))))
    return str(candidates[0])

def _walk_input_dirs():
    root = Path("/kaggle/input")
    if not root.exists():
        return
    for dirpath, dirnames, filenames in os.walk(root):
        yield Path(dirpath)

def _path_has_protenix_repo(path: str | Path | None) -> bool:
    if not path:
        return False
    p = Path(path)
    return (
        p.is_dir()
        and (p / "runner" / "inference.py").exists()
        and (p / "configs" / "configs_base.py").exists()
    )

def _path_has_protenix_runtime(path: str | Path | None, model_name: str) -> bool:
    if not path:
        return False
    p = Path(path)
    return (
        p.is_dir()
        and (p / "checkpoint" / f"{model_name}.pt").exists()
        and (p / "common" / "components.cif").exists()
        and (p / "common" / "components.cif.rdkit_mol.pkl").exists()
    )

def _rank_protenix_dir(path: Path):
    s = str(path).lower()
    return (
        "protenix" not in s,
        "kaggle" not in s,
        len(str(path)),
        str(path),
    )

def _walk_limited(base: Path, max_depth: int = 3):
    if not base.exists() or not base.is_dir():
        return
    base_depth = len(base.parts)
    for dirpath, dirnames, filenames in os.walk(base):
        p = Path(dirpath)
        depth = len(p.parts) - base_depth
        if depth > max_depth:
            dirnames[:] = []
            continue
        yield p

def _path_distance(a: str | Path | None, b: str | Path | None) -> int:
    if not a or not b:
        return 10**9
    pa = Path(a)
    pb = Path(b)
    a_parts = pa.parts
    b_parts = pb.parts
    common = 0
    for xa, xb in zip(a_parts, b_parts):
        if xa != xb:
            break
        common += 1
    return (len(a_parts) - common) + (len(b_parts) - common)

def _search_repo_near(anchor: str | Path | None) -> str | None:
    if not anchor:
        return None
    start = Path(anchor)
    probes = []
    if start.exists():
        probes.append(start)
    curr = start
    for _ in range(4):
        curr = curr.parent
        if curr.exists():
            probes.append(curr)
    candidates = []
    seen = set()
    for base in probes:
        for p in _walk_limited(base, max_depth=3) or []:
            key = str(p)
            if key in seen:
                continue
            seen.add(key)
            if _path_has_protenix_repo(p):
                candidates.append(p)
    if not candidates:
        return None
    candidates.sort(key=lambda p: (_path_distance(anchor, p),) + _rank_protenix_dir(p))
    return str(candidates[0])

def _search_runtime_near(anchor: str | Path | None, model_name: str) -> str | None:
    if not anchor:
        return None
    start = Path(anchor)
    probes = []
    if start.exists():
        probes.append(start)
    curr = start
    for _ in range(4):
        curr = curr.parent
        if curr.exists():
            probes.append(curr)
    candidates = []
    seen = set()
    for base in probes:
        for p in _walk_limited(base, max_depth=3) or []:
            key = str(p)
            if key in seen:
                continue
            seen.add(key)
            if _path_has_protenix_runtime(p, model_name):
                candidates.append(p)
    if not candidates:
        return None
    candidates.sort(key=lambda p: (_path_distance(anchor, p),) + _rank_protenix_dir(p))
    return str(candidates[0])

def _auto_find_protenix_code_dir() -> str | None:
    candidates = []
    for p in _walk_input_dirs() or []:
        if _path_has_protenix_repo(p):
            candidates.append(p)
    if not candidates:
        root = Path("/kaggle/input")
        if root.exists():
            for f in root.rglob("runner/inference.py"):
                cand = f.parent.parent
                if _path_has_protenix_repo(cand):
                    candidates.append(cand)
    if not candidates:
        return None
    uniq = sorted({str(p) for p in candidates})
    ranked = sorted((Path(p) for p in uniq), key=_rank_protenix_dir)
    return str(ranked[0])

def _auto_find_protenix_root(model_name: str) -> str | None:
    candidates = []
    for p in _walk_input_dirs() or []:
        if _path_has_protenix_runtime(p, model_name):
            candidates.append(p)
    if not candidates:
        return None
    uniq = sorted({str(p) for p in candidates})
    ranked = sorted((Path(p) for p in uniq), key=_rank_protenix_dir)
    return str(ranked[0])

def _finalize_protenix_paths(code_dir: str | None, root_dir: str | None, model_name: str):
    code_dir = str(code_dir) if code_dir else None
    root_dir = str(root_dir) if root_dir else None

    if not _path_has_protenix_runtime(root_dir, model_name):
        guessed_root = _search_runtime_near(code_dir, model_name) or _auto_find_protenix_root(model_name)
        if guessed_root is not None:
            root_dir = guessed_root

    if not _path_has_protenix_repo(code_dir):
        guessed_code = _search_repo_near(root_dir) or _auto_find_protenix_code_dir()
        if guessed_code is not None:
            code_dir = guessed_code

    if not _path_has_protenix_runtime(root_dir, model_name) and _path_has_protenix_runtime(code_dir, model_name):
        root_dir = code_dir
    if not _path_has_protenix_repo(code_dir) and _path_has_protenix_repo(root_dir):
        code_dir = root_dir

    return code_dir, root_dir

def _collect_protenix_hints(limit: int = 30) -> list[str]:
    hints = []
    seen = set()
    for p in _walk_input_dirs() or []:
        s = str(p)
        s_l = s.lower()
        interesting = (
            "protenix" in s_l
            or (p / "runner" / "inference.py").exists()
            or (p / "checkpoint").is_dir()
            or (p / "common").is_dir()
        )
        if interesting and s not in seen:
            seen.add(s)
            hints.append(s)
            if len(hints) >= limit:
                break
    hints.sort(key=lambda x: ("protenix" not in x.lower(), len(x), x))
    return hints[:limit]

def _format_protenix_missing_error(code_dir: str | None, root_dir: str | None, model_name: str) -> str:
    lines = [
        "Unable to resolve usable Protenix paths under /kaggle/input.",
        f"Resolved code_dir: {code_dir}",
        f"Resolved root_dir: {root_dir}",
        f"Needed repo marker: runner/inference.py + configs/configs_base.py",
        f"Needed runtime marker: checkpoint/{model_name}.pt + common/components.cif + common/components.cif.rdkit_mol.pkl",
        "Set PROTENIX_CODE_DIR / PROTENIX_ROOT_DIR explicitly or attach the Protenix repo+checkpoint dataset.",
    ]
    hints = _collect_protenix_hints(limit=20)
    if hints:
        lines.append("Interesting /kaggle/input paths detected:")
        lines.extend([f"  - {x}" for x in hints])
    else:
        lines.append("No Protenix-like directories were detected under /kaggle/input.")
    return "\n".join(lines)

def resolve_paths():
    data_dir = os.environ.get("COMP_DATA_DIR", COMP_DATA_DIR)
    if not Path(data_dir).exists():
        auto = _auto_find_competition_data()
        if auto is not None:
            data_dir = auto

    test_csv = os.environ.get("TEST_CSV", str(Path(data_dir) / "test_sequences.csv"))
    train_csv = os.environ.get("TRAIN_CSV", str(Path(data_dir) / "train_sequences.csv"))
    train_lbls = os.environ.get("TRAIN_LBLS", str(Path(data_dir) / "train_labels.csv"))
    val_csv = os.environ.get("VAL_CSV", str(Path(data_dir) / "validation_sequences.csv"))
    val_lbls = os.environ.get("VAL_LBLS", str(Path(data_dir) / "validation_labels.csv"))
    output_csv = os.environ.get("SUBMISSION_CSV", DEFAULT_OUTPUT)

    code_dir = os.environ.get("PROTENIX_CODE_DIR", DEFAULT_CODE_DIR)
    root_dir = os.environ.get("PROTENIX_ROOT_DIR", DEFAULT_ROOT_DIR)
    code_dir, root_dir = _finalize_protenix_paths(code_dir, root_dir, MODEL_NAME)

    return test_csv, train_csv, train_lbls, val_csv, val_lbls, output_csv, code_dir, root_dir

def ensure_required_files(root_dir: str) -> None:
    for p, name in [
        (Path(root_dir) / "checkpoint" / f"{MODEL_NAME}.pt", "checkpoint"),
        (Path(root_dir) / "common" / "components.cif", "CCD file"),
        (Path(root_dir) / "common" / "components.cif.rdkit_mol.pkl", "CCD cache"),
    ]:
        if not p.exists():
            raise FileNotFoundError(f"Missing {name}: {p}")

# ─────────────────────── Protenix Input / Config Helpers ─────────────────
def build_input_json(df: pd.DataFrame, json_path: str) -> None:
    data = [
        {
            "name": row["target_id"],
            "covalent_bonds": [],
            "sequences": [{"rnaSequence": {"sequence": row["sequence"], "count": 1}}],
        }
        for _, row in df.iterrows()
    ]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)

def build_configs(input_json_path: str, dump_dir: str, model_name: str):
    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from configs.configs_model_type import model_configs
    from protenix.config.config import parse_configs

    base = {**configs_base, **{"data": data_configs}, **inference_configs}

    def deep_update(t, p):
        for k, v in p.items():
            if isinstance(v, dict) and k in t and isinstance(t[k], dict):
                deep_update(t[k], v)
            else:
                t[k] = v

    deep_update(base, model_configs[model_name])
    arg_str = " ".join([
        f"--model_name {model_name}",
        f"--input_json_path {input_json_path}",
        f"--dump_dir {dump_dir}",
        f"--use_msa {USE_MSA}",
        f"--use_template {USE_TEMPLATE}",
        f"--use_rna_msa {USE_RNA_MSA}",
        f"--sample_diffusion.N_sample {N_SAMPLE}",
        f"--seeds {SEED}",
    ])
    return parse_configs(configs=base, arg_str=arg_str, fill_required_with_null=True)

# ───────────────────────── Submission Helpers ────────────────────────────
def coords_to_rows(target_id: str, seq: str, coords: np.ndarray) -> list:
    rows = []
    for i in range(len(seq)):
        row = {"ID": f"{target_id}_{i + 1}", "resname": seq[i], "resid": i + 1}
        for s in range(N_SAMPLE):
            if s < coords.shape[0] and i < coords.shape[1]:
                x, y, z = coords[s, i]
            else:
                x, y, z = 0.0, 0.0, 0.0
            row[f"x_{s + 1}"] = float(x)
            row[f"y_{s + 1}"] = float(y)
            row[f"z_{s + 1}"] = float(z)
        rows.append(row)
    return rows

def split_into_chunks(seq_len: int, max_len: int, overlap: int) -> list[tuple[int, int]]:
    if seq_len <= max_len:
        return [(0, seq_len)]
    chunks = []
    step = max_len - overlap
    pos = 0
    while pos < seq_len:
        end = min(pos + max_len, seq_len)
        chunks.append((pos, end))
        if end == seq_len:
            break
        pos += step
    return chunks

def kabsch_align(P: np.ndarray, Q: np.ndarray):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    Pc = P - centroid_P
    Qc = Q - centroid_Q

    # Fall back to translation-only if overlap is degenerate
    rank_p = np.linalg.matrix_rank(Pc) if len(P) >= 3 else 0
    rank_q = np.linalg.matrix_rank(Qc) if len(Q) >= 3 else 0
    if len(P) < 3 or len(Q) < 3 or min(rank_p, rank_q) < 2:
        return np.eye(3), centroid_Q - centroid_P

    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    S = np.eye(3)
    if d < 0:
        S[2, 2] = -1
    R = Vt.T @ S @ U.T
    t = centroid_Q - R @ centroid_P
    return R, t

def stitch_chunk_coords(chunk_coords_list: list,
                        chunk_ranges: list,
                        seq_len: int) -> np.ndarray:
    """
    Merge overlapping chunk coordinates into a full-sequence geometry.
    Aligns chunk i to chunk i-1 on the overlap, then uses linear ramps.
    """
    if len(chunk_coords_list) == 1:
        coords = np.asarray(chunk_coords_list[0], dtype=float)
        out = np.zeros((seq_len, 3), dtype=float)
        used = min(seq_len, coords.shape[0])
        out[:used] = coords[:used]
        return out

    aligned = [np.asarray(chunk_coords_list[0], dtype=float).copy()]

    for i in range(1, len(chunk_coords_list)):
        prev_start, prev_end = chunk_ranges[i - 1]
        cur_start, cur_end = chunk_ranges[i]
        ov_start = cur_start
        ov_end = min(prev_end, cur_end)
        ov_len = ov_end - ov_start

        cur = np.asarray(chunk_coords_list[i], dtype=float).copy()
        if ov_len < 3:
            aligned.append(cur)
            continue

        prev_ov = aligned[i - 1][ov_start - prev_start: ov_end - prev_start]
        cur_ov = cur[ov_start - cur_start: ov_end - cur_start]
        valid = ~(np.isnan(prev_ov).any(axis=1) | np.isnan(cur_ov).any(axis=1))

        if valid.sum() >= 3:
            R, t = kabsch_align(cur_ov[valid], prev_ov[valid])
            cur = (cur @ R.T) + t
        else:
            # translation-only on the overlap centroids if possible
            if valid.any():
                shift = prev_ov[valid].mean(axis=0) - cur_ov[valid].mean(axis=0)
                cur = cur + shift

        aligned.append(cur)

    full = np.zeros((seq_len, 3), dtype=np.float64)
    weights = np.zeros(seq_len, dtype=np.float64)

    for i, ((s, e), coords) in enumerate(zip(chunk_ranges, aligned)):
        actual_end = min(s + coords.shape[0], seq_len)
        used_len = actual_end - s
        w = np.ones(used_len, dtype=np.float64)

        if i > 0:
            ov_start = s
            ov_end = min(chunk_ranges[i - 1][1], e)
            ramp_len = ov_end - ov_start
            if ramp_len > 0:
                w[:ramp_len] = np.linspace(0.0, 1.0, ramp_len)

        if i < len(chunk_ranges) - 1:
            next_s = chunk_ranges[i + 1][0]
            ramp_start = max(0, next_s - s)
            ramp_len = max(0, actual_end - next_s)
            if ramp_len > 0 and ramp_start < used_len:
                w[ramp_start:used_len] = np.linspace(1.0, 0.0, ramp_len)

        full[s:actual_end] += coords[:used_len] * w[:, None]
        weights[s:actual_end] += w

    mask = weights > 0
    full[mask] /= weights[mask, None]
    return full

# ───────────────────────── Template Setup ────────────────────────────────
class _FallbackAlignment:
    def __init__(self, query_seq, template_seq):
        from difflib import SequenceMatcher
        self.query_seq = query_seq
        self.template_seq = template_seq
        sm = SequenceMatcher(None, query_seq, template_seq, autojunk=False)
        blocks = [b for b in sm.get_matching_blocks() if b.size > 0]
        self.aligned = (
            [(b.a, b.a + b.size) for b in blocks],
            [(b.b, b.b + b.size) for b in blocks],
        )
        matches = sum(b.size for b in blocks)
        len_penalty = 0.05 * abs(len(query_seq) - len(template_seq))
        self.score = 2.0 * matches - len_penalty

class _FallbackAligner:
    def __init__(self):
        self.mode = "global"
        self.match_score = 2.0
        self.mismatch_score = -1.5
        self.open_gap_score = -8.0
        self.extend_gap_score = -0.4

    @staticmethod
    def _kmer_set(seq, k=3):
        if len(seq) < k:
            return {seq} if seq else set()
        return {seq[i:i + k] for i in range(len(seq) - k + 1)}

    def score(self, query_seq, template_seq):
        min_len = min(len(query_seq), len(template_seq))
        if min_len == 0:
            return 0.0

        sample_n = min(64, min_len)
        idx = np.linspace(0, min_len - 1, sample_n).astype(int)
        pos_ident = float(np.mean([query_seq[i] == template_seq[i] for i in idx])) if len(idx) else 0.0

        q3 = self._kmer_set(query_seq, k=3)
        t3 = self._kmer_set(template_seq, k=3)
        jacc = len(q3 & t3) / max(1, len(q3 | t3)) if (q3 or t3) else 0.0

        length_ratio = min_len / max(len(query_seq), len(template_seq))
        sim = 0.60 * pos_ident + 0.40 * jacc
        return 2.0 * min_len * sim * length_ratio

    def align(self, query_seq, template_seq):
        return [_FallbackAlignment(query_seq, template_seq)]

def make_aligner():
    try:
        from Bio.Align import PairwiseAligner
        al = PairwiseAligner()
        al.mode = "global"
        al.match_score = 2
        al.mismatch_score = -1.5
        al.open_gap_score = -8
        al.extend_gap_score = -0.4
        al.query_left_open_gap_score = -8
        al.query_left_extend_gap_score = -0.4
        al.query_right_open_gap_score = -8
        al.query_right_extend_gap_score = -0.4
        al.target_left_open_gap_score = -8
        al.target_left_extend_gap_score = -0.4
        al.target_right_open_gap_score = -8
        al.target_right_extend_gap_score = -0.4
        print("Template aligner: Biopython PairwiseAligner")
        return al
    except Exception as e:
        print(f"Template aligner fallback active: {e}")
        return _FallbackAligner()

_aligner = make_aligner()

def parse_stoichiometry(stoich: str) -> list:
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    return [(ch.strip(), int(cnt)) for part in str(stoich).split(";") for ch, cnt in [part.split(":")]]

def parse_fasta(fasta_content: str) -> dict:
    out, cur, parts = {}, None, []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cur is not None:
                out[cur] = "".join(parts)
            cur = line[1:].split()[0]
            parts = []
        else:
            parts.append(line.replace(" ", ""))
    if cur is not None:
        out[cur] = "".join(parts)
    return out

def get_chain_segments(row) -> list:
    seq = row["sequence"]
    stoich = row.get("stoichiometry", "")
    all_sq = row.get("all_sequences", "")
    if pd.isna(stoich) or pd.isna(all_sq) or str(stoich).strip() == "" or str(all_sq).strip() == "":
        return [(0, len(seq))]
    try:
        chain_dict = parse_fasta(all_sq)
        order = parse_stoichiometry(stoich)
        segs, pos = [], 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                segs.append((pos, pos + len(base)))
                pos += len(base)
        return segs if pos == len(seq) else [(0, len(seq))]
    except Exception:
        return [(0, len(seq))]

def build_segments_map(df: pd.DataFrame) -> tuple[dict, dict]:
    seg_map, stoich_map = {}, {}
    for _, r in df.iterrows():
        tid = r["target_id"]
        seg_map[tid] = get_chain_segments(r)
        raw_s = r.get("stoichiometry", "")
        stoich_map[tid] = "" if pd.isna(raw_s) else str(raw_s)
    return seg_map, stoich_map

def process_labels(labels_df: pd.DataFrame) -> dict:
    coords = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for prefix, grp in labels_df.groupby(prefixes):
        coords[prefix] = grp.sort_values("resid")[["x_1", "y_1", "z_1"]].values
    return coords

def _build_aligned_strings(query_seq, template_seq, alignment):
    q_segs, t_segs = alignment.aligned
    aq, at, qi, ti = [], [], 0, 0
    for (qs, qe), (ts, te) in zip(q_segs, t_segs):
        while qi < qs:
            aq.append(query_seq[qi]); at.append("-"); qi += 1
        while ti < ts:
            aq.append("-"); at.append(template_seq[ti]); ti += 1
        for qp, tp in zip(range(qs, qe), range(ts, te)):
            aq.append(query_seq[qp]); at.append(template_seq[tp])
        qi, ti = qe, te
    while qi < len(query_seq):
        aq.append(query_seq[qi]); at.append("-"); qi += 1
    while ti < len(template_seq):
        aq.append("-"); at.append(template_seq[ti]); ti += 1
    return "".join(aq), "".join(at)

def find_similar_sequences(query_seq, train_seqs_df, train_coords_dict, top_n=24):
    similar = []
    for _, row in train_seqs_df.iterrows():
        tid, tseq = row["target_id"], row["sequence"]
        if tid not in train_coords_dict:
            continue
        if abs(len(tseq) - len(query_seq)) / max(len(tseq), len(query_seq)) > 0.35:
            continue
        raw_score = _aligner.score(query_seq, tseq)
        norm_s = raw_score / (2 * min(len(query_seq), len(tseq)))
        similar.append((tid, tseq, norm_s, train_coords_dict[tid]))
    similar.sort(key=lambda x: x[2], reverse=True)
    return similar[:top_n]

def find_similar_sequences_detailed(query_seq, train_seqs_df, train_coords_dict, top_n=12):
    shortlist = find_similar_sequences(
        query_seq=query_seq,
        train_seqs_df=train_seqs_df,
        train_coords_dict=train_coords_dict,
        top_n=max(top_n * 6, 24),
    )
    results = []
    for tid, tseq, _, tmpl_coords in shortlist:
        aln = next(iter(_aligner.align(query_seq, tseq)))
        norm_s = aln.score / (2 * min(len(query_seq), len(tseq)))
        identical = 0
        for (qs, qe), (ts, te) in zip(*aln.aligned):
            for qp, tp in zip(range(qs, qe), range(ts, te)):
                if query_seq[qp] == tseq[tp]:
                    identical += 1
        pct_id = 100 * identical / max(1, len(query_seq))
        aq, at = _build_aligned_strings(query_seq, tseq, aln)
        results.append((tid, tseq, norm_s, tmpl_coords, pct_id, aq, at))
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:top_n]

# ───────────────────────── Geometry Helpers ──────────────────────────────
def _estimate_direction_from_coords(coords: np.ndarray, idx: int) -> np.ndarray:
    left = [j for j in range(idx - 1, -1, -1) if np.isfinite(coords[j]).all()]
    right = [j for j in range(idx + 1, len(coords)) if np.isfinite(coords[j]).all()]
    if len(left) >= 2:
        v = coords[left[0]] - coords[left[1]]
        n = np.linalg.norm(v)
        if n > 1e-6:
            return v / n
    if len(right) >= 2:
        v = coords[right[0]] - coords[right[1]]
        n = np.linalg.norm(v)
        if n > 1e-6:
            return v / n
    return np.array([1.0, 0.0, 0.0], dtype=float)

def adapt_template_to_query(query_seq, template_seq, template_coords) -> np.ndarray:
    aln = next(iter(_aligner.align(query_seq, template_seq)))
    new_coords = np.full((len(query_seq), 3), np.nan, dtype=float)
    for (qs, qe), (ts, te) in zip(*aln.aligned):
        chunk = template_coords[ts:te]
        if len(chunk) == (qe - qs):
            new_coords[qs:qe] = chunk

    for i in range(len(new_coords)):
        if np.isnan(new_coords[i, 0]):
            pv = next((j for j in range(i - 1, -1, -1) if np.isfinite(new_coords[j]).all()), -1)
            nv = next((j for j in range(i + 1, len(new_coords)) if np.isfinite(new_coords[j]).all()), -1)
            if pv >= 0 and nv >= 0:
                w = (i - pv) / (nv - pv)
                new_coords[i] = (1 - w) * new_coords[pv] + w * new_coords[nv]
            elif pv >= 0:
                step = _estimate_direction_from_coords(new_coords, i) * 5.95
                new_coords[i] = new_coords[pv] + step * max(1, i - pv)
            elif nv >= 0:
                step = _estimate_direction_from_coords(new_coords, i) * 5.95
                new_coords[i] = new_coords[nv] - step * max(1, nv - i)
            else:
                new_coords[i] = np.array([i * 5.95, 0.0, 0.0], dtype=float)
    return np.nan_to_num(new_coords)

def adaptive_rna_constraints(coords, target_id, segments_map, confidence=1.0, passes=2) -> np.ndarray:
    X = np.asarray(coords, dtype=float).copy()
    segments = segments_map.get(target_id, [(0, len(X))])
    strength = max(0.75 * (1.0 - min(confidence, 0.97)), 0.02)
    for _ in range(passes):
        for s, e in segments:
            C = X[s:e]
            L = e - s
            if L < 3:
                continue
            d = C[1:] - C[:-1]
            dist = np.linalg.norm(d, axis=1) + 1e-6
            adj = d * ((5.95 - dist) / dist)[:, None] * (0.22 * strength)
            C[:-1] -= adj
            C[1:] += adj

            d2 = C[2:] - C[:-2]
            d2n = np.linalg.norm(d2, axis=1) + 1e-6
            adj2 = d2 * ((10.2 - d2n) / d2n)[:, None] * (0.10 * strength)
            C[:-2] -= adj2
            C[2:] += adj2

            C[1:-1] += (0.06 * strength) * (0.5 * (C[:-2] + C[2:]) - C[1:-1])

            if L >= 25:
                idx = np.linspace(0, L - 1, min(L, 160)).astype(int) if L > 220 else np.arange(L)
                P = C[idx]
                diff = P[:, None, :] - P[None, :, :]
                dm = np.linalg.norm(diff, axis=2) + 1e-6
                sep = np.abs(idx[:, None] - idx[None, :])
                mask = (sep > 2) & (dm < 3.2)
                if np.any(mask):
                    vec = (diff * ((3.2 - dm) / dm)[:, :, None] * mask[:, :, None]).sum(axis=1)
                    C[idx] += (0.015 * strength) * vec

            X[s:e] = C
    return X

def _rotmat(axis, ang):
    a = np.asarray(axis, float)
    a /= np.linalg.norm(a) + 1e-12
    x, y, z = a
    c, s = np.cos(ang), np.sin(ang)
    CC = 1 - c
    return np.array([
        [c + x * x * CC, x * y * CC - z * s, x * z * CC + y * s],
        [y * x * CC + z * s, c + y * y * CC, y * z * CC - x * s],
        [z * x * CC - y * s, z * y * CC + x * s, c + z * z * CC],
    ])

def apply_hinge(coords, seg, rng, max_angle_deg=18):
    s, e = seg
    L = e - s
    if L < 30:
        return np.asarray(coords, dtype=float).copy()
    pivot = s + int(rng.integers(10, L - 10))
    R = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg))))
    X = np.asarray(coords, dtype=float).copy()
    p0 = X[pivot].copy()
    X[pivot + 1:e] = (X[pivot + 1:e] - p0) @ R.T + p0
    return X

def jitter_chains(coords, segs, rng, max_angle_deg=10, max_trans=1.2):
    X = np.asarray(coords, dtype=float).copy()
    gc_ = X.mean(0, keepdims=True)
    for s, e in segs:
        R = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg))))
        shift = rng.normal(size=3)
        shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0, max_trans))
        c = X[s:e].mean(0, keepdims=True)
        X[s:e] = (X[s:e] - c) @ R.T + c + shift
    X -= X.mean(0, keepdims=True) - gc_
    return X

def smooth_wiggle(coords, segs, rng, amp=0.6):
    X = np.asarray(coords, dtype=float).copy()
    for s, e in segs:
        L = e - s
        if L < 20:
            continue
        ctrl = np.linspace(0, L - 1, 6)
        disp = rng.normal(0, amp, (6, 3))
        t = np.arange(L)
        X[s:e] += np.vstack([np.interp(t, ctrl, disp[:, k]) for k in range(3)]).T
    return X

def generate_rna_structure(sequence: str, seed=None) -> np.ndarray:
    if seed is not None:
        np.random.seed(seed)
    n = len(sequence)
    coords = np.zeros((n, 3), dtype=float)
    for i in range(n):
        ang = i * 0.6
        coords[i] = [10.0 * np.cos(ang), 10.0 * np.sin(ang), i * 2.5]
    return coords

# ───────────────────────── Sanitization ──────────────────────────────────
def sanitize_coords(coords: np.ndarray, target_id: str, segments_map: dict,
                    confidence: float = 0.5, clip: bool = True) -> np.ndarray:
    X = np.asarray(coords, dtype=float).copy()
    if X.ndim != 2 or X.shape[1] != 3:
        X = np.asarray(X, dtype=float).reshape(-1, 3)

    bad_abs = np.abs(X) > ABSURD_COORD_THRESHOLD
    X[bad_abs] = np.nan
    X[~np.isfinite(X)] = np.nan

    for s, e in segments_map.get(target_id, [(0, len(X))]):
        C = X[s:e].copy()
        L = len(C)
        if L == 0:
            continue

        valid = np.isfinite(C).all(axis=1)
        if not valid.any():
            X[s:e] = generate_rna_structure("A" * L)
            continue

        # Fill gaps by interpolation/extrapolation
        i = 0
        while i < L:
            if np.isfinite(C[i]).all():
                i += 1
                continue
            j = i
            while j < L and not np.isfinite(C[j]).all():
                j += 1

            prev_idx = i - 1
            next_idx = j if j < L else -1

            if prev_idx >= 0 and next_idx >= 0:
                for k in range(i, j):
                    w = (k - prev_idx) / (next_idx - prev_idx)
                    C[k] = (1 - w) * C[prev_idx] + w * C[next_idx]
            elif prev_idx >= 0:
                step = _estimate_direction_from_coords(C, prev_idx) * 5.95
                for k in range(i, j):
                    C[k] = C[k - 1] + step
            elif next_idx >= 0:
                step = _estimate_direction_from_coords(C, next_idx) * 5.95
                for k in range(j - 1, -1, -1):
                    if not np.isfinite(C[k]).all():
                        C[k] = C[k + 1] - step
            else:
                C[:] = generate_rna_structure("A" * L)
            i = j

        # Soft repair for any absurd chain jumps that remain
        for _ in range(2):
            d = np.linalg.norm(C[1:] - C[:-1], axis=1)
            bad_jump = np.where(d > 80.0)[0]
            if len(bad_jump) == 0:
                break
            for jj in bad_jump:
                if jj + 1 < L:
                    C[jj + 1] = 0.5 * (C[jj] + C[min(jj + 2, L - 1)])

        X[s:e] = C

    X = adaptive_rna_constraints(X, target_id, segments_map, confidence=confidence, passes=1)
    if clip:
        X = np.clip(X, COORD_CLIP_MIN, COORD_CLIP_MAX)
    return X

def sanitize_stack(pred_stack: np.ndarray, target_id: str, segments_map: dict,
                   confidence: float = 0.5) -> np.ndarray:
    pred_stack = np.asarray(pred_stack, dtype=float)
    out = []
    for i in range(pred_stack.shape[0]):
        out.append(sanitize_coords(pred_stack[i], target_id, segments_map, confidence=confidence))
    return np.stack(out, axis=0)

# ───────────────────────── Candidate Ranking ─────────────────────────────
def clamp01(x):
    return float(max(0.0, min(1.0, x)))

def compute_template_features(query_seq, template_seq, similarity, percent_identity,
                              aligned_query, aligned_template):
    aligned_len = max(1, len(aligned_query))
    matched = sum((q != "-" and t != "-") for q, t in zip(aligned_query, aligned_template))
    coverage = matched / max(1, len(query_seq))
    gap_fraction = 1.0 - (matched / aligned_len)
    length_ratio = min(len(query_seq), len(template_seq)) / max(len(query_seq), len(template_seq))

    confidence = (
        0.38 * clamp01(similarity) +
        0.27 * clamp01(percent_identity / 100.0) +
        0.22 * clamp01(coverage) +
        0.13 * clamp01(length_ratio) -
        0.10 * clamp01(gap_fraction)
    )
    confidence = clamp01(confidence)

    return {
        "coverage": coverage,
        "gap_fraction": gap_fraction,
        "length_ratio": length_ratio,
        "confidence": confidence,
    }

def build_template_prediction_candidates(sequence, target_id, train_seqs_df,
                                         train_coords_dict, segments_map,
                                         max_candidates=MAX_TEMPLATE_SEARCH):
    hits = find_similar_sequences_detailed(
        sequence,
        train_seqs_df,
        train_coords_dict,
        top_n=max_candidates,
    )

    candidates = []
    for rank, hit in enumerate(hits, start=1):
        tmpl_id, tmpl_seq, similarity, tmpl_coords, pct_id, aligned_q, aligned_t = hit
        feat = compute_template_features(sequence, tmpl_seq, similarity, pct_id, aligned_q, aligned_t)
        if feat["confidence"] < MIN_TEMPLATE_QUALITY_KEEP and pct_id < 20.0:
            continue

        coords = adapt_template_to_query(sequence, tmpl_seq, tmpl_coords)
        coords = sanitize_coords(coords, target_id, segments_map, confidence=feat["confidence"])

        candidates.append({
            "pred_id": f"template::{tmpl_id}",
            "source": "template",
            "template_id": tmpl_id,
            "similarity": float(similarity),
            "percent_identity": float(pct_id),
            "coverage": float(feat["coverage"]),
            "gap_fraction": float(feat["gap_fraction"]),
            "quality": float(feat["confidence"]),
            "coords": coords,
            "rank_hint": rank,
        })

    uniq = {}
    for c in sorted(candidates, key=lambda x: x["quality"], reverse=True):
        uniq.setdefault(c["template_id"], c)
    return list(uniq.values())[:max_candidates]

def route_from_template_candidates(candidate_meta, seq_len):
    if not candidate_meta:
        return {
            "route": "fallback_heavy",
            "template_slots": 0,
            "protenix_slots": N_SAMPLE,
            "best_confidence": 0.0,
            "strong_count": 0,
        }

    best_conf = candidate_meta[0]["quality"]
    strong_count = sum(c["quality"] >= TEMPLATE_STRONG_CONF for c in candidate_meta)
    medium_count = sum(c["quality"] >= TEMPLATE_MEDIUM_CONF for c in candidate_meta)

    if best_conf >= TEMPLATE_ONLY_CONF and strong_count >= 4:
        route = "template_only"
        template_slots, protenix_slots = N_SAMPLE, 0
    elif best_conf >= TEMPLATE_STRONG_CONF and medium_count >= 2:
        route = "hybrid"
        template_slots, protenix_slots = 3, 2
    elif best_conf >= TEMPLATE_MEDIUM_CONF:
        route = "hybrid"
        template_slots, protenix_slots = 2, 3
    elif best_conf >= TEMPLATE_WEAK_CONF:
        route = "fallback_heavy"
        template_slots, protenix_slots = 1, 4
    else:
        route = "fallback_heavy"
        template_slots, protenix_slots = 0, 5

    if seq_len > 800 and template_slots > 0:
        template_slots = max(template_slots, min(3, len(candidate_meta)))
        protenix_slots = max(0, N_SAMPLE - template_slots)

    return {
        "route": route,
        "template_slots": template_slots,
        "protenix_slots": protenix_slots,
        "best_confidence": best_conf,
        "strong_count": strong_count,
    }

def make_template_aug_candidates(best_candidate: dict, target_id: str, segments_map: dict,
                                 n_aug: int = 2, seed: int = 0) -> list[dict]:
    if best_candidate is None or n_aug <= 0:
        return []
    rng = np.random.default_rng(seed)
    segments = segments_map.get(target_id, [(0, len(best_candidate["coords"]))])
    coords = best_candidate["coords"]
    vars_ = []
    if n_aug >= 1:
        vars_.append(smooth_wiggle(coords, segments, rng, amp=0.45))
    if n_aug >= 2:
        vars_.append(apply_hinge(coords, max(segments, key=lambda se: se[1] - se[0]), rng, max_angle_deg=14))
    if n_aug >= 3:
        vars_.append(jitter_chains(coords, segments, rng, max_angle_deg=8, max_trans=0.8))

    out = []
    for i, v in enumerate(vars_[:n_aug]):
        out.append({
            "pred_id": f"template_aug::{best_candidate['template_id']}::{i}",
            "source": "template_aug",
            "template_id": best_candidate["template_id"],
            "similarity": best_candidate.get("similarity"),
            "percent_identity": best_candidate.get("percent_identity"),
            "quality": max(0.05, best_candidate["quality"] - 0.035 * (i + 1)),
            "coords": sanitize_coords(v, target_id, segments_map, confidence=best_candidate["quality"]),
        })
    return out

def make_prediction_aug_candidates(base_coords: np.ndarray, target_id: str, segments_map: dict,
                                   n_aug: int = 2, seed: int = 0,
                                   source_prefix: str = "protenix_aug",
                                   base_quality: float = 0.55) -> list[dict]:
    if n_aug <= 0:
        return []
    rng = np.random.default_rng(seed)
    segments = segments_map.get(target_id, [(0, len(base_coords))])
    vars_ = []
    if n_aug >= 1:
        vars_.append(smooth_wiggle(base_coords, segments, rng, amp=0.35))
    if n_aug >= 2:
        vars_.append(apply_hinge(base_coords, max(segments, key=lambda se: se[1] - se[0]), rng, max_angle_deg=10))
    if n_aug >= 3:
        vars_.append(jitter_chains(base_coords, segments, rng, max_angle_deg=6, max_trans=0.6))
    out = []
    for i, v in enumerate(vars_[:n_aug]):
        out.append({
            "pred_id": f"{source_prefix}::{i}",
            "source": source_prefix,
            "template_id": None,
            "similarity": None,
            "percent_identity": None,
            "quality": max(0.05, base_quality - 0.03 * (i + 1)),
            "coords": sanitize_coords(v, target_id, segments_map, confidence=base_quality),
        })
    return out

def make_de_novo_candidate(sequence, target_id, segments_map, variant_idx=0, seed=0):
    base = generate_rna_structure(sequence, seed=seed)
    rng = np.random.default_rng(seed)
    segments = segments_map.get(target_id, [(0, len(sequence))])

    if variant_idx == 1:
        base = smooth_wiggle(base, segments, rng, amp=0.8)
    elif variant_idx == 2:
        base = apply_hinge(base, max(segments, key=lambda se: se[1] - se[0]), rng, max_angle_deg=18)
    elif variant_idx >= 3:
        base = jitter_chains(base, segments, rng, max_angle_deg=10, max_trans=1.3)

    refined = sanitize_coords(base, target_id, segments_map, confidence=0.20)
    return {
        "pred_id": f"denovo::{variant_idx}",
        "source": "de_novo",
        "template_id": None,
        "similarity": None,
        "percent_identity": None,
        "quality": max(0.02, 0.18 - 0.02 * variant_idx),
        "coords": refined,
    }

def aligned_rmsd(coords_a, coords_b, max_points=500):
    A = np.asarray(coords_a, dtype=float)
    B = np.asarray(coords_b, dtype=float)
    n = min(len(A), len(B))
    A = A[:n]
    B = B[:n]
    if len(A) == 0:
        return 0.0
    if len(A) > max_points:
        idx = np.linspace(0, len(A) - 1, max_points).astype(int)
        A = A[idx]
        B = B[idx]
    R, t = kabsch_align(A, B)
    A_aligned = (A @ R.T) + t
    return float(np.sqrt(np.mean(np.sum((A_aligned - B) ** 2, axis=1))))

def select_final_five(candidates, n_out=N_SAMPLE):
    if not candidates:
        return []
    cands = [dict(c) for c in candidates]
    cands.sort(key=lambda x: x.get("quality", 0.0), reverse=True)
    selected = [cands[0]]
    remaining = cands[1:]

    while remaining and len(selected) < n_out:
        q_best = remaining[0].get("quality", 0.0)
        feasible = [c for c in remaining if c.get("quality", 0.0) >= q_best - QUALITY_DELTA]
        if not feasible:
            feasible = remaining[:]

        best_item = None
        best_score = -1e9
        for cand in feasible:
            if len(selected) == 1:
                bonus = 0.0
            else:
                div = min(aligned_rmsd(cand["coords"], s["coords"], max_points=RMSD_SUBSAMPLE) for s in selected)
                bonus = DIVERSITY_LAMBDA * min(div, DIVERSITY_CAP)
            score = cand.get("quality", 0.0) + bonus
            if score > best_score:
                best_score = score
                best_item = cand
        selected.append(best_item)
        remaining.remove(best_item)

    return selected[:n_out]

# ─────────────────────── Phase 1: Target Planning ────────────────────────
def plan_targets(test_df, train_seqs_df, train_coords_dict, segments_map):
    print(f"\n{'=' * 72}")
    print("PHASE 1: template search + routing")
    print(f"{'=' * 72}")

    plans = {}
    protenix_queue = {}

    for row_idx, row in test_df.reset_index(drop=True).iterrows():
        tid = row["target_id"]
        seq = row["sequence"]
        template_candidates = build_template_prediction_candidates(
            seq, tid, train_seqs_df, train_coords_dict, segments_map,
            max_candidates=MAX_TEMPLATE_CANDIDATES,
        )
        route_info = route_from_template_candidates(template_candidates, len(seq))
        plans[tid] = {
            "target_id": tid,
            "sequence": seq,
            "template_candidates": template_candidates,
            "route_info": route_info,
            "row_idx": row_idx,
        }

        if route_info["protenix_slots"] > 0:
            protenix_queue[tid] = {
                "n_needed": route_info["protenix_slots"],
                "sequence": seq,
            }

        if template_candidates:
            best = template_candidates[0]
            print(
                f"  {tid:>10s} | best={best['template_id']} "
                f"conf={best['quality']:.3f} sim={best['similarity']:.3f} "
                f"id={best['percent_identity']:.1f}% | "
                f"route={route_info['route']} t={route_info['template_slots']} p={route_info['protenix_slots']}"
            )
        else:
            print(
                f"  {tid:>10s} | no strong template | "
                f"route={route_info['route']} t={route_info['template_slots']} p={route_info['protenix_slots']}"
            )

    return plans, protenix_queue

# ─────────────────────── Phase 2: Protenix Inference ─────────────────────
def run_protenix_phase(protenix_queue: dict) -> dict:
    protenix_preds = {}
    if not protenix_queue:
        return protenix_preds

    if not USE_PROTENIX:
        print(f"\nPHASE 2 skipped (USE_PROTENIX=False).")
        return protenix_preds

    print(f"\n{'=' * 72}")
    print(f"PHASE 2: Protenix for {len(protenix_queue)} routed targets")
    print(f"{'=' * 72}")

    work_dir = Path("/kaggle/working")
    work_dir.mkdir(parents=True, exist_ok=True)

    tasks = []
    chunk_info = {}
    infer_slots_map = {}

    for target_id, info in protenix_queue.items():
        full_seq = info["sequence"]
        seq_len = len(full_seq)
        n_needed = int(info["n_needed"])
        n_infer = n_needed if seq_len <= MAX_SEQ_LEN else min(n_needed, MAX_LONG_SAMPLES)
        infer_slots_map[target_id] = n_infer

        if seq_len <= MAX_SEQ_LEN:
            tasks.append({"target_id": target_id, "sequence": full_seq})
            chunk_info[target_id] = [{"name": target_id, "range": (0, seq_len)}]
            print(f"  {target_id} ({seq_len} nt): single pass ×{n_infer}")
        else:
            chunks = split_into_chunks(seq_len, MAX_SEQ_LEN, CHUNK_OVERLAP)
            print(f"  {target_id} ({seq_len} nt): {len(chunks)} chunks ×{n_infer}")
            chunk_info[target_id] = []
            for ci, (cs, ce) in enumerate(chunks):
                chunk_name = f"{target_id}_chunk{ci}"
                tasks.append({"target_id": chunk_name, "sequence": full_seq[cs:ce]})
                chunk_info[target_id].append({"name": chunk_name, "range": (cs, ce)})

    if not tasks:
        return protenix_preds

    tasks_df = pd.DataFrame(tasks)
    input_json_path = str(work_dir / "protenix_queue_input.json")
    build_input_json(tasks_df, input_json_path)

    import sys
    q_path = "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"
    if q_path not in sys.path:
        sys.path.insert(0, q_path)
        
    try:
        import biotite.structure.io.pdbx.convert as pdbx_convert
        if not hasattr(pdbx_convert, "PDBX_BOND_TYPE_ID_TO_TYPE"):
            pdbx_convert.PDBX_BOND_TYPE_ID_TO_TYPE = {}
    except Exception:
        pass
    # ---------------------------------
    
    from protenix.data.inference.infer_dataloader import InferenceDataset
    from runner.inference import InferenceRunner, update_gpu_compatible_configs, update_inference_configs
    # ---------------------------------------

    configs = build_configs(input_json_path, str(work_dir / "outputs"), MODEL_NAME)
    configs = update_gpu_compatible_configs(configs)
    runner = InferenceRunner(configs)
    dataset = InferenceDataset(configs)

    raw_predictions = {}

    def _extract_c1_coords(data, raw_coords):
        feat = data["input_feature_dict"]
        chunk_seq_len = int(data["N_token"].item())
        if "centre_atom_mask" in feat:
            mask = (feat["centre_atom_mask"] == 1).to(raw_coords.device)
        elif "atom_to_tokatom_idx" in feat:
            m11 = (feat["atom_to_tokatom_idx"] == 11).to(raw_coords.device)
            m12 = (feat["atom_to_tokatom_idx"] == 12).to(raw_coords.device)
            c11, c12 = int(m11.sum()), int(m12.sum())
            mask = m11 if abs(c11 - chunk_seq_len) < abs(c12 - chunk_seq_len) else m12
        else:
            mask = torch.zeros(raw_coords.shape[1], dtype=torch.bool, device=raw_coords.device)

        coords = raw_coords[:, mask, :].detach().cpu().numpy()
        if coords.shape[1] == 0:
            return None

        if coords.shape[1] > 1:
            diffs = np.linalg.norm(coords[0, 1:] - coords[0, :-1], axis=-1)
            if np.all(diffs < 1e-4):
                return None

        if coords.shape[1] != chunk_seq_len:
            if coords.shape[1] == 1 and chunk_seq_len > 1:
                return None
            padded = np.zeros((coords.shape[0], chunk_seq_len, 3), dtype=np.float32)
            ml = min(coords.shape[1], chunk_seq_len)
            padded[:, :ml, :] = coords[:, :ml, :]
            coords = padded

        return coords

    for i in tqdm(range(len(dataset)), desc="Protenix Inference"):
        data, atom_array, err = dataset[i]
        sample_name = data.get("sample_name", f"sample_{i}")
        if err:
            print(f"  {sample_name} data error: {err}")
            raw_predictions[sample_name] = None
            try:
                del data, atom_array, err
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue

        target_id = sample_name.split("_chunk")[0] if "_chunk" in sample_name else sample_name
        n_infer = infer_slots_map.get(target_id, N_SAMPLE)
        sub_seq_len = int(data["N_token"].item())

        try:
            new_cfg = update_inference_configs(configs, sub_seq_len)
            new_cfg.sample_diffusion.N_sample = n_infer
            runner.update_model_configs(new_cfg)
            pred = runner.predict(data)
            raw_coords = pred["coordinate"]
            coords = _extract_c1_coords(data, raw_coords)
            raw_predictions[sample_name] = coords
        except Exception as exc:
            print(f"  {sample_name} inference failed: {exc}")
            raw_predictions[sample_name] = None
        finally:
            for obj_name in ["pred", "data", "atom_array", "raw_coords"]:
                try:
                    del locals()[obj_name]
                except Exception:
                    pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    for target_id, info in protenix_queue.items():
        full_seq = info["sequence"]
        seq_len = len(full_seq)
        n_infer = infer_slots_map[target_id]
        chunks = chunk_info.get(target_id, [])

        if not chunks:
            continue

        if len(chunks) == 1:
            coords = raw_predictions.get(target_id)
            if coords is not None:
                coords = sanitize_stack(coords, target_id, GLOBAL_SEGMENTS_MAP, confidence=SHORT_PTX_BASE_QUALITY)
                protenix_preds[target_id] = coords
                print(f"  {target_id}: {coords.shape[0]} predictions generated")
            else:
                protenix_preds[target_id] = None
                print(f"  {target_id}: FAILED")
        else:
            chunk_results_per_sample = {s: [] for s in range(n_infer)}
            all_ok = True
            for cinfo in chunks:
                cname = cinfo["name"]
                crange = cinfo["range"]
                ccoords = raw_predictions.get(cname)
                if ccoords is None:
                    all_ok = False
                    break
                for s_idx in range(n_infer):
                    if s_idx < ccoords.shape[0]:
                        chunk_results_per_sample[s_idx].append((ccoords[s_idx], crange))
                    else:
                        chunk_results_per_sample[s_idx].append((ccoords[-1], crange))

            if not all_ok:
                print(f"  {target_id}: chunked inference incomplete")
                protenix_preds[target_id] = None
                continue

            stitched_samples = []
            for s_idx in range(n_infer):
                items = chunk_results_per_sample[s_idx]
                coords_list = [c for c, _ in items]
                ranges_list = [r for _, r in items]
                full_coords = stitch_chunk_coords(coords_list, ranges_list, seq_len)
                stitched_samples.append(full_coords)

            result = sanitize_stack(np.stack(stitched_samples, axis=0), target_id, GLOBAL_SEGMENTS_MAP,
                                    confidence=LONG_PTX_BASE_QUALITY)
            protenix_preds[target_id] = result
            print(f"  {target_id}: {result.shape[0]} stitched predictions generated")

    return protenix_preds

# ─────────────────────── Phase 3: Final Assembly ────────────────────────
def assemble_submission(test_df: pd.DataFrame, plans: dict, protenix_preds: dict, segments_map: dict) -> pd.DataFrame:
    print(f"\n{'=' * 72}")
    print("PHASE 3: final candidate selection + submission writeout")
    print(f"{'=' * 72}")

    all_rows = []

    for row_idx, row in test_df.reset_index(drop=True).iterrows():
        tid = row["target_id"]
        seq = row["sequence"]
        plan = plans[tid]
        route_info = plan["route_info"]
        template_candidates = list(plan["template_candidates"])

        candidate_pool = list(template_candidates)

        if template_candidates and len(template_candidates) < route_info["template_slots"]:
            needed_aug = min(AUGMENT_TEMPLATE_VARIANTS, route_info["template_slots"] - len(template_candidates))
            candidate_pool.extend(
                make_template_aug_candidates(
                    template_candidates[0],
                    tid,
                    segments_map,
                    n_aug=needed_aug,
                    seed=(row_idx + 1) * 1009 + 17,
                )
            )

        ptx = protenix_preds.get(tid)
        if ptx is not None and ptx.ndim == 3:
            base_q = SHORT_PTX_BASE_QUALITY if len(seq) <= MAX_SEQ_LEN else LONG_PTX_BASE_QUALITY
            for j in range(ptx.shape[0]):
                candidate_pool.append({
                    "pred_id": f"protenix::{j}",
                    "source": "protenix" if len(seq) <= MAX_SEQ_LEN else "protenix_chunked",
                    "template_id": None,
                    "similarity": None,
                    "percent_identity": None,
                    "quality": max(0.05, base_q - 0.03 * j),
                    "coords": ptx[j],
                })

            # If the route asked for more Protenix-like diversity than we could infer,
            # add light perturbations of the best Protenix sample instead of de-novo.
            need_ptx_like = max(0, route_info["protenix_slots"] - ptx.shape[0])
            if need_ptx_like > 0:
                candidate_pool.extend(
                    make_prediction_aug_candidates(
                        ptx[0], tid, segments_map,
                        n_aug=min(need_ptx_like, 3),
                        seed=(row_idx + 1) * 7919 + 11,
                        source_prefix="protenix_aug",
                        base_quality=base_q - 0.01,
                    )
                )

        denovo_idx = 0
        while len(candidate_pool) < max(N_SAMPLE, 7):
            candidate_pool.append(
                make_de_novo_candidate(
                    seq, tid, segments_map,
                    variant_idx=denovo_idx,
                    seed=(row_idx + 1) * 1000003 + denovo_idx * 7919,
                )
            )
            denovo_idx += 1

        final_candidates = select_final_five(candidate_pool, n_out=N_SAMPLE)
        while len(final_candidates) < N_SAMPLE:
            final_candidates.append(
                make_de_novo_candidate(
                    seq, tid, segments_map,
                    variant_idx=len(final_candidates),
                    seed=(row_idx + 1) * 1409 + len(final_candidates) * 97,
                )
            )

        stacked = np.stack([
            sanitize_coords(c["coords"], tid, segments_map, confidence=c.get("quality", 0.5))
            for c in final_candidates[:N_SAMPLE]
        ], axis=0)

        source_summary = ", ".join(f"{i + 1}:{c['source']}({c['quality']:.3f})" for i, c in enumerate(final_candidates[:N_SAMPLE]))
        print(f"  {tid}: {source_summary}")

        all_rows.extend(coords_to_rows(tid, seq, stacked))

    sub = pd.DataFrame(all_rows)
    cols = ["ID", "resname", "resid"] + [f"{c}_{i}" for i in range(1, N_SAMPLE + 1) for c in ["x", "y", "z"]]
    coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]
    sub[coord_cols] = sub[coord_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    sub[coord_cols] = sub[coord_cols].clip(COORD_CLIP_MIN, COORD_CLIP_MAX)
    return sub[cols]

# ───────────────────────── Main ───────────────────────────────────────────
GLOBAL_SEGMENTS_MAP = {}

def main() -> None:
    global GLOBAL_SEGMENTS_MAP

    import sys
    sys.path.insert(0, "/kaggle/input/protenix-rmsa-repo/protenix_kaggle")

    test_csv, train_csv, train_lbls, val_csv, val_lbls, output_csv, code_dir, root_dir = resolve_paths()

    print("Resolved paths")
    print("  test_csv :", test_csv)
    print("  train_csv:", train_csv)
    print("  val_csv  :", val_csv)
    print("  code_dir :", code_dir)
    print("  root_dir :", root_dir)
    print("  USE_PROTENIX:", USE_PROTENIX)
    print("  MAX_SEQ_LEN / CHUNK_OVERLAP:", MAX_SEQ_LEN, "/", CHUNK_OVERLAP)

    if not Path(test_csv).exists():
        raise FileNotFoundError(f"Missing test CSV: {test_csv}")

    if USE_PROTENIX:
        code_dir, root_dir = _finalize_protenix_paths(code_dir, root_dir, MODEL_NAME)
        print("  code_dir (final):", code_dir)
        print("  root_dir (final):", root_dir)
        if not _path_has_protenix_repo(code_dir) or not _path_has_protenix_runtime(root_dir, MODEL_NAME):
            raise FileNotFoundError(_format_protenix_missing_error(code_dir, root_dir, MODEL_NAME))
        os.environ["PROTENIX_ROOT_DIR"] = root_dir
        if code_dir not in sys.path:
            sys.path.append(code_dir)
        ensure_required_files(root_dir)

    seed_everything(SEED)

    test_df_full = pd.read_csv(test_csv)
    test_df = (test_df_full.head(LOCAL_N_SAMPLES) if not IS_KAGGLE and LOCAL_N_SAMPLES else test_df_full).reset_index(drop=True)
    print(f"\nTest targets: {len(test_df)}" + (" (LOCAL MODE)" if not IS_KAGGLE else ""))

    print("\nLoading train/validation template bank …")
    train_seqs = pd.read_csv(train_csv)
    val_seqs = pd.read_csv(val_csv)
    train_labels = pd.read_csv(train_lbls)
    val_labels = pd.read_csv(val_lbls)

    combined_seqs = pd.concat([train_seqs, val_seqs], ignore_index=True)
    combined_labels = pd.concat([train_labels, val_labels], ignore_index=True)
    train_coords = process_labels(combined_labels)
    GLOBAL_SEGMENTS_MAP, _ = build_segments_map(test_df)

    print(f"Template pool: {len(combined_seqs)} sequences, {len(train_coords)} structures")

    plans, protenix_queue = plan_targets(test_df, combined_seqs, train_coords, GLOBAL_SEGMENTS_MAP)
    protenix_preds = run_protenix_phase(protenix_queue)
    sub = assemble_submission(test_df, plans, protenix_preds, GLOBAL_SEGMENTS_MAP)

    output_path = Path(output_csv)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(output_path, index=False)

    coord_cols = [c for c in sub.columns if c.startswith(("x_", "y_", "z_"))]
    absurd_count = int(((sub[coord_cols].abs() > ABSURD_COORD_THRESHOLD).any(axis=1)).sum())
    nan_count = int(sub[coord_cols].isna().sum().sum())

    print(f"\n✓ Saved submission to {output_path} ({len(sub):,} rows)")
    print(f"  NaN coords after save : {nan_count}")
    print(f"  Absurd coords > {ABSURD_COORD_THRESHOLD:g}: {absurd_count}")
    print(sub.head())


In [ ]:
if __name__ == "__main__":
    main()